# Cavity overlap factor for HFGW detection in a static-$B$ cylindrical cavity (Berlin et al. 2022)

This notebook implements a **numerical, scan-ready** calculation of the normalized cavity overlap/form factor
\[
\eta \equiv \frac{\int_{V_{\rm cav}} dV\,\mathbf E_{\rm mode}^*(\mathbf x)\cdot \mathbf j_{\rm eff}(\mathbf x)}{\sqrt{V_{\rm cav}}\,\sqrt{\int_{V_{\rm cav}} dV\,|\mathbf E_{\rm mode}(\mathbf x)|^2}},
\]
for a cylindrical cavity (radius $a$, length $L$) with static background magnetic field $\mathbf B_0=B_0\hat z$.

We follow the spirit and notation of Berlin et al., *Phys. Rev. D 105, 116011 (2022)*, with a deliberate implementation choice:

1. Keep the GW tensor structure in the **canonical reference configuration** (GW propagating along reference $\hat z$).
2. Encode arbitrary GW arrival direction $(\alpha,\beta)$ by rotating the **background EM/Faraday tensor** into that GW frame.
3. Build the effective current from these fixed GW tensors + rotated background tensor.

So directional dependence enters through the rotated background field, **not** from re-deriving a fully off-axis GW polarization tensor in lab coordinates.

The notebook includes:
- TE/TM cylindrical cavity modes,
- proper detector-frame finite-wavelength kernels $F_0,F_1,F_2$ (stable near zero argument),
- overlap integrals on a cylindrical mesh,
- angle scans and antenna-pattern plots.


In [ ]:
# --- User-editable parameters ---
import numpy as np
import matplotlib.pyplot as plt
from scipy import special

# Physical / geometry
a = 0.10               # cavity radius [m]
L = 0.50               # cavity length [m]
B0 = 5.0               # static background B-field magnitude [T]
omega_g = 2*np.pi*10e9 # GW angular frequency [rad/s]
c = 299792458.0

# Mode selection
mode_family = "TM"    # "TM" or "TE"
m, n, p = 0, 1, 1

# Polarization basis setup
# pol_basis: "plus", "cross", or "linear" (mix by psi)
pol_basis = "linear"
psi = np.deg2rad(0.0)  # polarization angle for linear combination

# Direction grids (GW direction relative to cavity frame)
# beta: polar angle from cavity +z ; alpha: azimuth
beta_grid = np.linspace(0, np.pi, 73)
alpha_grid = np.linspace(0, 2*np.pi, 97)

# Spatial grid
Nr, Nphi, Nz = 70, 72, 90

# Flags
plot_fields = True
plot_antenna = True
run_validations = True


## 1) Coordinate conventions, mesh, and utilities

We use cavity cylindrical coordinates $(r,\phi,z)$ with
\[
r\in[0,a],\quad \phi\in[0,2\pi),\quad z\in[-L/2,L/2].
\]
Mesh arrays use `indexing='ij'` consistently.

Direction convention:
\[
\hat n(\alpha,\beta)=\begin{bmatrix}\cos\alpha\sin\beta\\\sin\alpha\sin\beta\\\cos\beta\end{bmatrix}
\]
is the GW propagation direction in the cavity frame. In this notebook we keep GW tensors canonical along reference $\hat z$ and rotate background tensors by the inverse frame map.


In [ ]:
def make_cylindrical_grid(a, L, Nr, Nphi, Nz):
    r = np.linspace(0.0, a, Nr)
    phi = np.linspace(0.0, 2*np.pi, Nphi, endpoint=False)
    z = np.linspace(-L/2, L/2, Nz)
    dr = r[1]-r[0] if Nr > 1 else a
    dphi = phi[1]-phi[0] if Nphi > 1 else 2*np.pi
    dz = z[1]-z[0] if Nz > 1 else L

    R, PHI, Z = np.meshgrid(r, phi, z, indexing='ij')
    X = R*np.cos(PHI)
    Y = R*np.sin(PHI)
    return {
        'r': r, 'phi': phi, 'z': z,
        'R': R, 'PHI': PHI, 'Z': Z,
        'X': X, 'Y': Y,
        'dr': dr, 'dphi': dphi, 'dz': dz
    }

def cyl_to_cart(vr, vphi, vz, phi):
    vx = vr*np.cos(phi) - vphi*np.sin(phi)
    vy = vr*np.sin(phi) + vphi*np.cos(phi)
    return vx, vy, vz

def cart_to_cyl(vx, vy, vz, phi):
    vr = vx*np.cos(phi) + vy*np.sin(phi)
    vphi = -vx*np.sin(phi) + vy*np.cos(phi)
    return vr, vphi, vz

mesh = make_cylindrical_grid(a, L, Nr, Nphi, Nz)
Vcav = np.pi*a*a*L


## 2) Cavity TE/TM eigenmodes in a cylinder

We use standard PEC cylindrical-cavity mode structure:
- TM$_{mnp}$ uses $\chi_{mn}$ zeros of $J_m$,
- TE$_{mnp}$ uses $\chi'_{mn}$ zeros of $J_m'$.

Define
\[
\gamma_{mn}=\frac{\chi_{mn}}{a}\ \text{(TM)},\qquad \gamma_{mn}=\frac{\chi'_{mn}}{a}\ \text{(TE)},\qquad k_z=\frac{p\pi}{L},
\]
\[
\omega_{mnp}=c\sqrt{\gamma_{mn}^2+k_z^2}.
\]

Below we return **raw field shapes** (not normalized), and compute normalization integrals separately.


In [ ]:
_mode_cache = {}

def mode_constants(mode_family, m, n, p, a, L, c=299792458.0):
    key = (mode_family, m, n, p, a, L)
    if key in _mode_cache:
        return _mode_cache[key]

    if mode_family.upper() == 'TM':
        chi = special.jn_zeros(m, n)[-1]
    elif mode_family.upper() == 'TE':
        chi = special.jnp_zeros(m, n)[-1]
    else:
        raise ValueError("mode_family must be 'TM' or 'TE'")

    gamma = chi / a
    kz = p*np.pi / L
    omega = c*np.sqrt(gamma**2 + kz**2)
    out = {'chi': chi, 'gamma': gamma, 'kz': kz, 'omega': omega}
    _mode_cache[key] = out
    return out

def _safe_div(num, den, eps=1e-14):
    out = np.zeros_like(num, dtype=np.complex128)
    mask = np.abs(den) > eps
    out[mask] = num[mask]/den[mask]
    return out

def cavity_mode_fields(mode_family, m, n, p, mesh, a, L):
    """Return raw electric field components (E_r, E_phi, E_z) on mesh.
    Time dependence convention e^{-i omega t}; complex spatial shape returned.
    """
    R, PHI, Z = mesh['R'], mesh['PHI'], mesh['Z']
    cst = mode_constants(mode_family, m, n, p, a, L)
    gamma, kz = cst['gamma'], cst['kz']

    # Common factors
    arg = gamma*R
    cos_mphi = np.cos(m*PHI)
    sin_mphi = np.sin(m*PHI)
    Zc = np.cos(kz*(Z + L/2))
    Zs = np.sin(kz*(Z + L/2))

    Jm = special.jv(m, arg)
    Jm_p = special.jvp(m, arg, 1)

    if mode_family.upper() == 'TM':
        # Use E_z generating form; tangential E vanishes at PEC walls through J_m(chi)=0 and z dependence.
        Ez = Jm * cos_mphi * Zs

        # Transverse from scalar potential derivatives (TM-like ansatz)
        # E_r ~ -(kz/gamma) J'_m cos(mphi) cos(kz z),
        # E_phi ~ +(kz/gamma) (m/(gamma r)) J_m sin(mphi) cos(kz z)
        Er = -(kz/gamma) * Jm_p * cos_mphi * Zc
        ratio = np.zeros_like(R)
        mask = R > 0
        ratio[mask] = (m/(gamma*R[mask])) * Jm[mask]
        if m == 0:
            ratio[~mask] = 0.0
        else:
            # small-r limit m>0 finite for m=1; tends to 0 for m>1
            ratio[~mask] = 0.0
        Ephi = +(kz/gamma) * ratio * sin_mphi * Zc

    else:  # TE mode: E_z = 0
        Ez = np.zeros_like(R, dtype=np.float64)

        # TE-like transverse electric pattern from H_z generator
        # E_r ~ -(m/(gamma r)) J_m sin(mphi) cos(kz z)
        # E_phi ~ -J'_m cos(mphi) cos(kz z)
        ratio = np.zeros_like(R)
        mask = R > 0
        ratio[mask] = (m/(gamma*R[mask])) * Jm[mask]
        if m == 0:
            ratio[~mask] = 0.0
        else:
            ratio[~mask] = 0.0

        Er = -ratio * sin_mphi * Zc
        Ephi = -Jm_p * cos_mphi * Zc

    return Er.astype(np.complex128), Ephi.astype(np.complex128), Ez.astype(np.complex128), cst


## 3) GW reference tensors and rotation strategy

### Canonical GW tensors
We define reference basis vectors
\[
\hat x'=(1,0,0),\quad \hat y'=(0,1,0),\quad \hat z'=(0,0,1)
\]
with GW propagation along $\hat z'$. Canonical TT tensors:
\[
e^+_0 = \hat x'\otimes\hat x' - \hat y'\otimes\hat y',\qquad
e^\times_0 = \hat x'\otimes\hat y' + \hat y'\otimes\hat x'.
\]
Polarization angle rotation:
\[
e^+(\psi)=\cos2\psi\,e^+_0+\sin2\psi\,e^\times_0,
\]
\[
e^\times(\psi)=-\sin2\psi\,e^+_0+\cos2\psi\,e^\times_0.
\]

### Rotation convention used here
Let $R(\alpha,\beta)=R_z(\alpha)R_y(\beta)$ map reference-frame vectors into cavity-frame vectors. We keep GW tensors in reference frame and rotate cavity background vectors/tensors into that frame by
\[
\mathbf B_{\rm ref}=R^T\mathbf B_{\rm cav},\qquad F_{\rm ref}=\Lambda^T F_{\rm cav}\Lambda
\]
(with purely spatial rotation here). This is the core implementation choice.


In [ ]:
def Rz(alpha):
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([[ca, -sa, 0.0], [sa, ca, 0.0], [0.0, 0.0, 1.0]])

def Ry(beta):
    cb, sb = np.cos(beta), np.sin(beta)
    return np.array([[cb, 0.0, sb], [0.0, 1.0, 0.0], [-sb, 0.0, cb]])

def rotation_matrix(alpha, beta):
    # reference -> cavity (active)
    return Rz(alpha) @ Ry(beta)

def reference_gw_polarization_tensors(psi=0.0):
    eplus0 = np.array([[1.,0.,0.],[0.,-1.,0.],[0.,0.,0.]])
    ecross0 = np.array([[0.,1.,0.],[1.,0.,0.],[0.,0.,0.]])
    c2, s2 = np.cos(2*psi), np.sin(2*psi)
    eplus = c2*eplus0 + s2*ecross0
    ecross = -s2*eplus0 + c2*ecross0
    return eplus, ecross


## 4) Background $B$-field and Faraday tensor handling

In cavity frame: $\mathbf E_0=0$, $\mathbf B_0=B_0\hat z$. We store the antisymmetric Faraday tensor with convention
\[
F_{0i}=E_i=0,\qquad F_{ij}=-\epsilon_{ijk}B_k.
\]
Under pure spatial rotations, $F$ rotates as a rank-2 tensor.


In [ ]:
def faraday_from_fields(E, B):
    F = np.zeros((4,4), dtype=float)
    F[0,1:], F[1:,0] = E, -np.array(E)
    Bx, By, Bz = B
    F[1,2] = -Bz; F[2,1] = +Bz
    F[1,3] = +By; F[3,1] = -By
    F[2,3] = -Bx; F[3,2] = +Bx
    return F

def fields_from_faraday(F):
    E = np.array([F[0,1], F[0,2], F[0,3]])
    Bx = -F[2,3]
    By = F[1,3]
    Bz = -F[1,2]
    return E, np.array([Bx, By, Bz])

def rotate_faraday_tensor(F_cav, alpha, beta):
    R = rotation_matrix(alpha, beta)        # ref->cav
    Lam = np.eye(4)
    Lam[1:,1:] = R
    # cavity -> reference is transpose for orthogonal rotation
    F_ref = Lam.T @ F_cav @ Lam
    return F_ref

def rotate_background_vectors(B0, alpha, beta):
    R = rotation_matrix(alpha, beta)
    B_cav = np.array([0.0, 0.0, B0])
    B_ref = R.T @ B_cav
    return B_ref

E0_cav = np.array([0.0,0.0,0.0])
B0_cav = np.array([0.0,0.0,B0])
F0_cav = faraday_from_fields(E0_cav, B0_cav)


## 5) Finite-wavelength kernels $F_0,F_1,F_2$ (proper detector frame)

To keep numerics stable we implement kernel functions in terms of
\[
q \equiv \frac{\omega_g z}{c}
\]
in the **reference $z$-propagation setup**.

We use smooth functions with well-defined small-$q$ limits:
\[
F_0(q)=\frac{\sin q}{q},\quad
F_1(q)=\frac{\sin q-q\cos q}{q^3},\quad
F_2(q)=\frac{2q\sin q+(2-q^2)\cos q-2}{q^4},
\]
with series branches near $q=0$.

These kernels capture finite-size longitudinal response structure and are used as numerically robust Berlin-style weighting functions in the source construction.


In [ ]:
def F0_kernel(q):
    q = np.asarray(q, dtype=float)
    out = np.empty_like(q)
    small = np.abs(q) < 1e-4
    qs = q[small]
    out[small] = 1 - qs**2/6 + qs**4/120
    out[~small] = np.sin(q[~small])/q[~small]
    return out

def F1_kernel(q):
    q = np.asarray(q, dtype=float)
    out = np.empty_like(q)
    small = np.abs(q) < 1e-3
    qs = q[small]
    # 1/3 - q^2/30 + q^4/840 + ...
    out[small] = 1/3 - qs**2/30 + qs**4/840
    qq = q[~small]
    out[~small] = (np.sin(qq) - qq*np.cos(qq))/qq**3
    return out

def F2_kernel(q):
    q = np.asarray(q, dtype=float)
    out = np.empty_like(q)
    small = np.abs(q) < 1e-3
    qs = q[small]
    # -1/12 + q^2/180 - q^4/6720 + ...
    out[small] = -1/12 + qs**2/180 - qs**4/6720
    qq = q[~small]
    out[~small] = (2*qq*np.sin(qq) + (2-qq**2)*np.cos(qq) - 2)/qq**4
    return out


## 6) Effective current source in tensor form (reference GW frame)

We implement a compact tensorized current model that keeps GW tensors fixed in reference form and injects incidence-angle dependence via rotated background field:

\[
\mathbf j_{\rm eff}(\mathbf x) = i\,\omega_g\Big[F_0(q)\,\mathbf T_0 + F_1(q)\,\mathbf T_1 + F_2(q)\,\mathbf T_2\Big],
\]
with $q=\omega_g z/c$ and
\[
\mathbf T_0 = h\cdot\mathbf B_{\rm ref},\quad
\mathbf T_1 = \hat z\,(\hat z\cdot h\cdot\mathbf B_{\rm ref}),\quad
\mathbf T_2 = h\cdot\hat z\,(\hat z\cdot\mathbf B_{\rm ref}).
\]

This preserves the notebook's principal design:
- canonical GW tensor basis,
- angle dependence only through rotated $\mathbf B$ / Faraday tensor.

The overall amplitude normalization can be rescaled depending on the exact Berlin convention; here we focus on consistent normalized overlap $\eta$.


In [ ]:
def metric_response_tensors(psi=0.0):
    eplus, ecross = reference_gw_polarization_tensors(psi=psi)
    return {'plus': eplus, 'cross': ecross}

def effective_current_cartesian(mesh, alpha, beta, psi=0.0, pol='plus', omega_g=1.0, B0=1.0, c=299792458.0):
    """Compute j_eff on mesh in Cartesian components in reference GW frame.
    Direction enters via rotated background B (equiv. Faraday rotation).
    """
    Z = mesh['Z']
    q = omega_g*Z/c
    f0, f1, f2 = F0_kernel(q), F1_kernel(q), F2_kernel(q)

    B_ref = rotate_background_vectors(B0, alpha, beta)
    hplus, hcross = reference_gw_polarization_tensors(psi=psi)

    if pol == 'plus':
        h = hplus
    elif pol == 'cross':
        h = hcross
    elif pol == 'linear':
        # linear combination controlled by psi already in basis rotation
        h = hplus
    else:
        raise ValueError("pol must be 'plus', 'cross', or 'linear'")

    ez = np.array([0.0,0.0,1.0])
    T0 = h @ B_ref
    z_h_B = ez @ (h @ B_ref)
    T1 = ez * z_h_B
    T2 = (h @ ez) * (ez @ B_ref)

    # Broadcast vector terms over mesh
    jx = 1j*omega_g*(f0*T0[0] + f1*T1[0] + f2*T2[0])
    jy = 1j*omega_g*(f0*T0[1] + f1*T1[1] + f2*T2[1])
    jz = 1j*omega_g*(f0*T0[2] + f1*T1[2] + f2*T2[2])

    return jx, jy, jz


## 7) Overlap integral and $\eta$

For each mode we compute
\[
I = \int dV\,\mathbf E^*_{\rm mode}\cdot\mathbf j_{\rm eff},\qquad
N_E = \int dV\,|\mathbf E_{\rm mode}|^2,
\]
then
\[
\eta = \frac{I}{\sqrt{V_{\rm cav}}\sqrt{N_E}}.
\]

We evaluate with cylindrical Jacobian $dV=r\,dr\,d\phi\,dz$.


In [ ]:
def volume_integral_cyl(integrand, mesh):
    R = mesh['R']
    return np.sum(integrand * R) * mesh['dr'] * mesh['dphi'] * mesh['dz']

def mode_fields_cartesian(mode_family, m, n, p, mesh, a, L):
    Er, Ephi, Ez, cst = cavity_mode_fields(mode_family, m, n, p, mesh, a, L)
    Ex, Ey, Ez = cyl_to_cart(Er, Ephi, Ez, mesh['PHI'])
    return Ex, Ey, Ez, cst

def compute_eta(mode_family, m, n, p, mesh, a, L, alpha, beta, psi,
                omega_g, B0, pol='plus', c=299792458.0):
    Ex, Ey, Ez, cst = mode_fields_cartesian(mode_family, m, n, p, mesh, a, L)
    jx, jy, jz = effective_current_cartesian(mesh, alpha, beta, psi=psi, pol=pol, omega_g=omega_g, B0=B0, c=c)

    overlap_density = np.conjugate(Ex)*jx + np.conjugate(Ey)*jy + np.conjugate(Ez)*jz
    I = volume_integral_cyl(overlap_density, mesh)

    Enorm_density = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    NE = np.real(volume_integral_cyl(Enorm_density, mesh))

    eta = I / (np.sqrt(Vcav)*np.sqrt(NE))
    return eta, I, NE, cst

def compute_eta_plus_cross(mode_family, m, n, p, mesh, a, L, alpha, beta, psi,
                           omega_g, B0, c=299792458.0):
    eta_p, *_ = compute_eta(mode_family, m, n, p, mesh, a, L, alpha, beta, psi, omega_g, B0, pol='plus', c=c)
    eta_x, *_ = compute_eta(mode_family, m, n, p, mesh, a, L, alpha, beta, psi, omega_g, B0, pol='cross', c=c)
    return eta_p, eta_x


## 8) Validation / sanity checks


In [ ]:
if run_validations:
    print("Kernel small-q checks:")
    q_test = np.array([0.0, 1e-6, 1e-4, 1e-2, 0.1])
    print("q:", q_test)
    print("F0:", F0_kernel(q_test))
    print("F1:", F1_kernel(q_test))
    print("F2:", F2_kernel(q_test))

    # Rotation consistency at alpha=beta=0
    B_ref_00 = rotate_background_vectors(B0, 0.0, 0.0)
    print("\nB_ref(alpha=0,beta=0) =", B_ref_00, "(should be [0,0,B0])")

    # On-axis vs tiny-angle continuity
    eta_on, I_on, NE_on, cst = compute_eta(mode_family, m, n, p, mesh, a, L,
                                           0.0, 0.0, psi, omega_g, B0, pol='plus', c=c)
    eta_tiny, *_ = compute_eta(mode_family, m, n, p, mesh, a, L,
                               1e-5, 1e-5, psi, omega_g, B0, pol='plus', c=c)
    print(f"\nMode omega_mnp/(2pi) = {cst['omega']/(2*np.pi):.6e} Hz")
    print("NE (mode norm integral) =", NE_on)
    print("eta_plus on-axis         =", eta_on)
    print("eta_plus tiny-angle      =", eta_tiny)
    print("|difference|             =", abs(eta_on-eta_tiny))

    # Symmetry hint for axisymmetric mode with m=0: alpha independence should be weak
    if m == 0:
        e1, *_ = compute_eta(mode_family, m, n, p, mesh, a, L,
                             0.0, np.pi/4, psi, omega_g, B0, pol='plus', c=c)
        e2, *_ = compute_eta(mode_family, m, n, p, mesh, a, L,
                             np.pi/3, np.pi/4, psi, omega_g, B0, pol='plus', c=c)
        print("\nFor m=0 benchmark, alpha sensitivity check:")
        print("eta(alpha=0,beta=pi/4)    =", e1)
        print("eta(alpha=pi/3,beta=pi/4) =", e2)
        print("|difference|              =", abs(e1-e2))


## 9) Quick field visualization for selected mode


In [ ]:
if plot_fields:
    Ex, Ey, Ez, cst = mode_fields_cartesian(mode_family, m, n, p, mesh, a, L)
    Emag = np.sqrt(np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2)

    iz = np.argmin(np.abs(mesh['z']))
    fig, ax = plt.subplots(1, 1, figsize=(6,5))
    pcm = ax.pcolormesh(mesh['X'][:,:,iz], mesh['Y'][:,:,iz], Emag[:,:,iz], shading='auto')
    ax.set_aspect('equal')
    ax.set_title(f"|E_mode| at z≈0 for {mode_family}_{m}{n}{p}")
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    plt.colorbar(pcm, ax=ax, label='arb.')
    plt.show()


## 10) Diagnostics: rotated background components


In [ ]:
beta_diag = np.linspace(0, np.pi, 181)
alpha_diag = 0.0
Bdiag = np.array([rotate_background_vectors(B0, alpha_diag, b) for b in beta_diag])

fig, ax = plt.subplots(1,1, figsize=(7,4))
ax.plot(beta_diag, Bdiag[:,0], label='B_ref,x')
ax.plot(beta_diag, Bdiag[:,1], label='B_ref,y')
ax.plot(beta_diag, Bdiag[:,2], label='B_ref,z')
ax.set_xlabel(r'$\beta$ [rad]')
ax.set_ylabel('component [T]')
ax.set_title(r'Rotated background field components ($\alpha=0$)')
ax.legend()
ax.grid(alpha=0.3)
plt.show()


## 11) Angle scans and antenna patterns


In [ ]:
def scan_eta_over_angles(mode_family, m, n, p, mesh, a, L,
                         alpha_vals, beta_vals, psi,
                         omega_g, B0, pol='plus', c=299792458.0):
    # loops over angle grid (dominant cost is 3D overlap each point)
    out = np.zeros((len(alpha_vals), len(beta_vals)), dtype=np.complex128)
    for ia, alpha in enumerate(alpha_vals):
        for ib, beta in enumerate(beta_vals):
            out[ia, ib], *_ = compute_eta(mode_family, m, n, p, mesh, a, L,
                                          alpha, beta, psi, omega_g, B0, pol=pol, c=c)
    return out

if plot_antenna:
    # 1D eta(beta) for fixed alpha, psi
    alpha0 = 0.0
    eta_beta_plus = np.array([
        compute_eta(mode_family, m, n, p, mesh, a, L, alpha0, b, psi, omega_g, B0, pol='plus', c=c)[0]
        for b in beta_grid
    ])
    eta_beta_cross = np.array([
        compute_eta(mode_family, m, n, p, mesh, a, L, alpha0, b, psi, omega_g, B0, pol='cross', c=c)[0]
        for b in beta_grid
    ])

    fig, ax = plt.subplots(1,1, figsize=(7,4))
    ax.plot(beta_grid, np.abs(eta_beta_plus), label='|eta_plus|')
    ax.plot(beta_grid, np.abs(eta_beta_cross), label='|eta_cross|')
    ax.set_xlabel(r'$\beta$ [rad]')
    ax.set_ylabel(r'$|\eta|$')
    ax.set_title(rf'{mode_family}_{m}{n}{p}: response vs polar angle ($\alpha=0$)')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

    # Polar plot
    fig = plt.figure(figsize=(6,6))
    pax = fig.add_subplot(111, projection='polar')
    pax.plot(beta_grid, np.abs(eta_beta_plus), label='plus')
    pax.plot(beta_grid, np.abs(eta_beta_cross), label='cross')
    pax.set_title(rf'Polar antenna pattern: {mode_family}_{m}{n}{p}')
    pax.legend(loc='upper right', bbox_to_anchor=(1.25,1.15))
    plt.show()

    # 2D map |eta(alpha,beta)| for fixed psi
    eta_map_plus = scan_eta_over_angles(mode_family, m, n, p, mesh, a, L,
                                        alpha_grid, beta_grid, psi, omega_g, B0, pol='plus', c=c)
    eta_map_cross = scan_eta_over_angles(mode_family, m, n, p, mesh, a, L,
                                         alpha_grid, beta_grid, psi, omega_g, B0, pol='cross', c=c)

    A, B = np.meshgrid(alpha_grid, beta_grid, indexing='ij')

    fig, axs = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
    im0 = axs[0].pcolormesh(B, A, np.abs(eta_map_plus), shading='auto')
    axs[0].set_xlabel(r'$\beta$ [rad]'); axs[0].set_ylabel(r'$\alpha$ [rad]')
    axs[0].set_title(r'$|\eta_+ (\alpha,\beta)|$')
    plt.colorbar(im0, ax=axs[0])

    im1 = axs[1].pcolormesh(B, A, np.abs(eta_map_cross), shading='auto')
    axs[1].set_xlabel(r'$\beta$ [rad]'); axs[1].set_ylabel(r'$\alpha$ [rad]')
    axs[1].set_title(r'$|\eta_\times (\alpha,\beta)|$')
    plt.colorbar(im1, ax=axs[1])
    plt.show()


## 12) Performance notes

- Bessel roots are cached in `_mode_cache`.
- Mode fields are evaluated once per overlap call; for large angle scans you can precompute mode fields and norm once and only update $\mathbf j_{\rm eff}$.
- The current source uses broadcasted array math; kernels are vectorized.
- Angle-grid scans are trivially parallelizable over `(alpha,beta)` points.
- The rotation-based strategy avoids expensive symbolic/coordinate re-derivation of off-axis GW tensors for every direction.


## Final summary

Implemented in this notebook:

1. Cylindrical-cavity TE/TM mode fields on 3D cylindrical grids.
2. Canonical reference GW polarization tensors (propagation fixed to reference $\hat z$).
3. Arbitrary incidence handled by rotating the background field/Faraday tensor into the GW reference frame.
4. Finite-wavelength kernels $F_0,F_1,F_2$ with stable small-argument branches.
5. Effective current source $\mathbf j_{\rm eff}$ assembled in Cartesian tensor form.
6. Normalized overlap/form factor $\eta$ and angle-scan tools for antenna patterns.
7. Validation diagnostics and plotting pipeline.

Conventions/approximations:
- The notebook keeps GW tensors canonical and puts directional dependence in rotated background tensors by design.
- Normalization of $\eta$ is explicit and consistent for relative sensitivity mapping; an overall prefactor may be adapted to exact Berlin amplitude conventions.
- Integrals are numerical on finite grids (convergence can be checked by increasing `Nr,Nphi,Nz`).

Natural next steps:
- automated mode scans across $(m,n,p)$,
- multimode directional reconstruction,
- quantitative side-by-side comparison to specific Berlin et al. figures,
- extension to SRF cavities and non-cylindrical geometries.
